In [1]:
# 품목별 추천재고량

# pip install psycopg2-binary pandas xgboost scikit-learn

import psycopg2
import pandas as pd
import numpy as np
from itertools import product
from datetime import datetime
import xgboost as xgb
from xgboost import XGBRegressor

# 경고 메시지 무시 (깔끔한 출력을 위해)
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 임포트 완료")

라이브러리 임포트 완료


In [8]:
def get_demand_forecast_data():
    """
    DB에서 수요 이력과 품목 정보를 조인하여 가져오는 함수
    """
    try:
        # DB 연결 설정
        conn = psycopg2.connect(
            host="115.178.75.126",
            port="5432",
            user="postgres",
            password="postgres123",
            database="postgres"
        )
        cursor = conn.cursor()
        
        # 쿼리 실행 (수요테이블 + 품목정보테이블)
        query = """
        SELECT
            A.SYS_ID, A.BIZ_SEQ, B.ITEM_NO,
            TO_CHAR(A.CLM_DT, 'YYYY-MM') AS YYYYMM,
            SUM(A.CLM_QTY) AS DEMAND_QTY,
            MAX(B.PRCHS_PNDG_PRD) AS LEAD_TIME,
            MAX(B.UNTPRC) AS UNIT_PRICE
        FROM BT_PRCHS_MTNC_SITU A
        INNER JOIN BT_ITEM_INFO_MNG B
            ON A.SYS_ID = B.SYS_ID AND A.BIZ_SEQ = B.BIZ_SEQ AND A.PRTS_NO = B.PRTS_NO
        WHERE A.CLM_DT IS NOT NULL
        GROUP BY A.SYS_ID, A.BIZ_SEQ, B.ITEM_NO, TO_CHAR(A.CLM_DT, 'YYYY-MM')
        ORDER BY YYYYMM ASC
        """
        
        cursor.execute(query)
        data = cursor.fetchall()
        cols = ['SYS_ID', 'BIZ_SEQ', 'ITEM_NO', 'YYYYMM', 'DEMAND_QTY', 'LEAD_TIME', 'UNIT_PRICE']
        
        df = pd.DataFrame(data, columns=cols)
        conn.close()
        
        print(f"DB 데이터 조회 성공: 총 {len(df)}건")
        return df
        
    except Exception as e:
        print(f"DB 연결 또는 조회 실패: {e}")
        return None

In [ ]:
def preprocess_demand_forecast_data(df):
    """
    1. 빈 월(Month)을 0으로 채우기 (Zero-Filling)
    2. 숫자형 변환 (Decimal -> Float)
    3. 파생변수(Feature) 생성 (이동평균, 표준편차 등)
    4. 정답(Target) 생성
    """
    if df is None or len(df) == 0:
        return None, None

    # (1) 날짜 포맷 변환
    df['DT'] = pd.to_datetime(df['YYYYMM'], format='%Y-%m')
    
    # (2) 전체 기간 뼈대 만들기
    min_date = df['DT'].min()
    max_date = df['DT'].max()
    max_date_extended = max_date + pd.DateOffset(months=1)
    
    all_months = pd.date_range(start=min_date, end=max_date_extended, freq='MS')
    unique_items = df['ITEM_NO'].unique()
    
    cartesian_df = pd.DataFrame(list(product(unique_items, all_months)), columns=['ITEM_NO', 'DT'])
    
    # (3) 원본 데이터와 병합
    df_merged = pd.merge(cartesian_df, df, on=['ITEM_NO', 'DT'], how='left')
    
    # (4) 결측치 및 데이터 타입 정리
    df_merged['DEMAND_QTY'] = df_merged['DEMAND_QTY'].fillna(0) 
    
    for col in ['SYS_ID', 'BIZ_SEQ', 'LEAD_TIME', 'UNIT_PRICE']:
        df_merged[col] = df_merged.groupby('ITEM_NO')[col].ffill().bfill()

    df_merged['UNIT_PRICE'] = pd.to_numeric(df_merged['UNIT_PRICE'], errors='coerce').fillna(0)
    df_merged['LEAD_TIME'] = pd.to_numeric(df_merged['LEAD_TIME'], errors='coerce').fillna(0)
    df_merged['DEMAND_QTY'] = pd.to_numeric(df_merged['DEMAND_QTY'], errors='coerce').fillna(0)
        
    # (5) Feature Engineering
    df_final = df_merged.sort_values(['ITEM_NO', 'DT'])
    
    # Lag Features
    df_final['PREV_1M'] = df_final.groupby('ITEM_NO')['DEMAND_QTY'].shift(1)
    df_final['AVG_3M'] = df_final.groupby('ITEM_NO')['DEMAND_QTY'].rolling(3, min_periods=1).mean().reset_index(0, drop=True)
    
    # [추가] 수요의 변동성(표준편차) 계산 -> 안전재고 산출용
    df_final['STD_3M'] = df_final.groupby('ITEM_NO')['DEMAND_QTY'].rolling(3, min_periods=1).std().reset_index(0, drop=True).fillna(0)
    
    # Target
    df_final['TARGET'] = df_final.groupby('ITEM_NO')['DEMAND_QTY'].shift(-1)
    
    df_final['YYYYMM'] = df_final['DT'].dt.strftime('%Y-%m')
    
    df_train = df_final.dropna(subset=['TARGET'])
    df_predict = df_final[df_final['TARGET'].isna()].copy()
    
    print(f"전처리 완료: 학습 데이터 {len(df_train)}건, 예측 대상 {len(df_predict)}건")
    return df_train, df_predict

In [ ]:
# ---------------------------------------------------------
# 데이터 로드 -> 전처리 -> 모델 학습 -> 추천재고량 산출
# ---------------------------------------------------------
